In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA notebook_breweries;

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

###Creazione Tabella gold

In [0]:
%skip
spark.sql("drop table if exists gold_breweries")

In [0]:
silver_df = spark.read.table("notebook_breweries.silver_breweries")

gold_table = "gold_breweries"

run_ts = spark.sql("SELECT current_timestamp()").collect()[0][0]

if not spark.catalog.tableExists(gold_table):
  (silver_df
          .withColumn(
                "brewery_sk", 
                F.sha2(
                        F.concat_ws(
                                "_",
                                F.col("id"),
                                F.col("ingestion_ts").cast("string")
                                ), 256
                        )
                ) #oppure possibilità di utilizzare F.expr("uuid()") - identificatore univoco casuale in formato 36 caratteri - ma non consigliato, in quanto ogni rieseczione
                  #genera una brewery_sk diversa per lo stesso record
          .withColumn("valid_from", F.col("ingestion_ts"))
          .withColumn("valid_to", F.lit(None).cast("timestamp"))
          .withColumn("current", F.lit(True))
          .write
          .format("delta")
          .mode("overwrite")
          .saveAsTable(gold_table)
  )


In [0]:
%skip 
spark.table("gold_breweries").printSchema()


###Merge SCD2 Manuale

In [0]:
delta_table_gold = DeltaTable.forName(spark, gold_table)

change_condition = """
    NOT (
    target.name <=> source.name AND
    target.brewery_type <=> source.brewery_type AND
    target.address_1 <=> source.address_1 AND
    target.address_2 <=> source.address_2 AND
    target.address_3 <=> source.address_3 AND
    target.city <=> source.city AND
    target.country <=> source.country AND
    target.latitude <=> source.latitude AND
    target.longitude <=> source.longitude AND
    target.phone <=> source.phone AND
    target.postal_code <=> source.postal_code AND
    target.state <=> source.state AND
    target.state_province <=> source.state_province AND
    target.street <=> source.street AND
    target.website_url <=> source.website_url
      )
    """
# 1) righe "changed" (solo quelle che matchano un current e sono diverse)
change_df = (
  silver_df.alias("source")
    .join(
      delta_table_gold.toDF().alias("target"),
      (F.col("target.id") == F.col("source.id")) &
      (F.col("target.current") == F.lit(True)),
      "inner"
    )
    .where(F.expr(change_condition))
    .select(*[F.col(f"source.{c}") for c in silver_df.columns])
)

# 2) staged: tutte le source + copia delle changed con merge_key NULL per forzare insert
staged_df = (
  silver_df.withColumn("merge_id", F.col("id"))
    .unionByName(
      change_df.withColumn("merge_id", F.lit(None).cast("string") )
    )
)
# 3) Merge
(
  delta_table_gold.alias("target")
  .merge(
    staged_df.alias("source"),
    "target.id = source.merge_id AND target.current = true"
  )
  .whenMatchedUpdate(
    condition = change_condition,
    set = {
      "valid_to": F.col("source.ingestion_ts"),
      "current": F.lit(False)
      }
    )
  .whenNotMatchedInsert(	
    values={
      "brewery_sk": F.sha2(
                        F.concat_ws(
                                "_",
                                F.col("id"),
                                F.col("ingestion_ts").cast("string")
                                ), 256
                        ),
      "id": F.col("source.id"),
      "name": F.col("source.name"),
      "brewery_type": F.col("source.brewery_type"),
      "address_1": F.col("source.address_1"),
      "address_2": F.col("source.address_2"),
      "address_3": F.col("source.address_3"),
      "city": F.col("source.city"),
      "country": F.col("source.country"),
      "latitude": F.col("source.latitude"),
      "longitude": F.col("source.longitude"),
      "phone": F.col("source.phone"),
      "postal_code": F.col("source.postal_code"),
      "state": F.col("source.state"),
      "state_province": F.col("source.state_province"),
      "street": F.col("source.street"),
      "website_url": F.col("source.website_url"),
      "valid_from": F.col("source.ingestion_ts"),
      "valid_to": F.lit(None).cast("timestamp"),
      "current": F.lit(True)
      }
  )
  .execute()
)

In [0]:
%skip
selezione = spark.sql("""
SELECT *
FROM gold_breweries
""")

display(selezione)

In [0]:
%skip
selezione = spark.sql("""
SELECT id,
       count_if(current = true) as current_rows,
       min(valid_from) as min_vf,
       max(valid_to) as max_vt
FROM gold_breweries
GROUP BY id
HAVING current_rows <> 1
""")

display(selezione)

In [0]:
%skip
selezione = spark.sql("""
SELECT id, street, valid_from, valid_to, current
FROM gold_breweries
WHERE id = '4dcaeaa3-d7cc-4016-9392-5bde4e3a8f4d'
ORDER BY valid_from
""")

display(selezione)

In [0]:
gold_table_order = "gold_breweries"

(
spark
    .read
    .table(gold_table_order)
    .select(
        "brewery_sk",          # surrogate prima
        "id",
        "name",
        "brewery_type",
        "address_1",
        "address_2",
        "address_3",
        "city",
        "state",
        "country",
        "postal_code",
        "phone",
        "website_url",
        "valid_from",
        "valid_to",
        "current",
        "latitude",
        "longitude",
        "state_province",
        "street"
    )
    .orderBy(F.col("brewery_sk").asc())
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

###Verfica SCD2

In [0]:
%skip
df_gold = spark.read.table(gold_table_order)
display(
    df_gold
        #.filter(F.col("current") == True)
        .where(F.col("id") == "4dcaeaa3-d7cc-4016-9392-5bde4e3a8f4d")
        .orderBy(F.col("brewery_sk").asc())
)

###Aggregazioni

####Numero Brewery per Stato

In [0]:
(
  spark.table(gold_table_order)
      .filter("current = true")
      .groupBy("state")
      .count()
      .withColumnRenamed("count", "num_breweries")
      .orderBy(F.desc("num_breweries"))
      .write
      .mode("overwrite")
      .format("delta")
      .saveAsTable("gold_table_breweries_by_state")
)

In [0]:
%skip
gold_selection = spark.sql("""SELECT * FROM gold_breweries""")
display(gold_selection)

In [0]:
%skip
gold_selection_aggregate = spark.sql("""SELECT * FROM gold_table_breweries_by_state""")
display(gold_selection_aggregate)